# E9-Fresh: NLBSE 2027 reference-style Colab T4 evaluation

**Goal:** independently evaluate the released E9 model using the settings of the organizers' [reference baseline notebook](https://colab.research.google.com/drive/1q1fGPpU7uxaK1mUVHa3b2xJOtEwhWwLG), rather than reusing the earlier 256-token Kaggle scores.

**Use Google Colab -> Runtime -> Change runtime type -> T4 GPU.** Internet access is required. This notebook never trains the model or accesses hidden-test labels.

The organizers' reference baseline notebook uses **512-token** maximum sequence length, **global padding**, batch size **600**, **autocast**, 10 repeated forward passes per batch, PyTorch profiler FLOPs, and these score constants: maximum average runtime = 1000 seconds; maximum average GFLOPs = 3,000,000. Their reference baseline notebook does not synchronize CUDA around its timing. Our version synchronizes for a more reliable elapsed-time value, and may decrease batch size automatically after an out-of-memory exception. Therefore **the resulting local efficiency score is an estimate, not the organizers' official ranking**.

The public Hugging Face dataset calls the 30,655-row labeled evaluation split `test`. On the competition website it is described as **public validation**. The 61,311-row organizer hidden test is unavailable.

In [ ]:
%pip -q install "transformers>=4.45,<6" "datasets>=3,<5" "scikit-learn>=1.4" safetensors

In [ ]:
from pathlib import Path
import gc, json, platform, time
from contextlib import nullcontext

import numpy as np
import torch
from datasets import load_dataset
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from IPython.display import FileLink, display

MODEL_ID = "naimNB/nlbse2027-e9-codebert"
DATASET_ID = "NLBSE/nlbse27-code-comment-classification"
SPLIT = "test"   # HF dataset name; public validation in the competition terminology
MAX_LENGTH = 512  # organizers' reference notebook; prior E9 tests used 256
REPETITIONS = 10
REFERENCE_BATCH_SIZE = 600
BATCH_CANDIDATES = [600, 128, 64, 32, 16, 8]   # fallback if T4 memory is insufficient
OUTPUT = Path("/content/E9_Colab_T4_reference_style_metrics.json")

assert torch.cuda.is_available(), "Colab GPU runtime is required. Choose T4 GPU."
GPU_NAME = torch.cuda.get_device_name(0)
print("GPU:", GPU_NAME)
print("PyTorch:", torch.__version__)
print("Transformers, datasets and other packages are imported.")
if "T4" not in GPU_NAME:
    print("WARNING: Not an NVIDIA T4; runtime is not hardware-comparable to organizer setup.")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID).to("cuda").eval()
assert model.config.num_labels == 2

test = load_dataset(DATASET_ID, split=SPLIT)
functions = [str(x or "") for x in test["Function"]]
targets = np.asarray(test["Label"], dtype=np.int64)
assert len(functions) == 30655 and targets.shape == (30655, 2), (len(functions), targets.shape)

# The reference notebook tokenizes all rows simultaneously with padding=True.
tokens = tokenizer(functions, return_tensors="pt", padding=True,
                   truncation=True, max_length=MAX_LENGTH)
input_ids, attention_mask = tokens["input_ids"], tokens["attention_mask"]
PADDED_LENGTH = int(input_ids.shape[1])
print("Examples:", len(functions), "| Global padded length:", PADDED_LENGTH)
print("Baseline reference maximum token length:", MAX_LENGTH)
del functions, tokens
gc.collect()

def autocast_context():
    return torch.autocast(device_type="cuda", dtype=torch.float16)

# Test candidate batches before launching the full repeated inference.
BATCH_SIZE = None
for candidate in BATCH_CANDIDATES:
    try:
        with torch.inference_mode(), autocast_context():
            _ = model(input_ids=input_ids[:candidate].to("cuda"),
                      attention_mask=attention_mask[:candidate].to("cuda")).logits
        torch.cuda.synchronize()
        del _
        BATCH_SIZE = candidate
        print("Batch size selected:", BATCH_SIZE)
        break
    except torch.cuda.OutOfMemoryError:
        print("OOM with batch size", candidate, "- trying smaller batch.")
        gc.collect()
        torch.cuda.empty_cache()

if BATCH_SIZE is None:
    raise RuntimeError("Even batch size 8 failed. Check GPU memory or restart runtime.")
if BATCH_SIZE != REFERENCE_BATCH_SIZE:
    print("NOTE: Batch differs from organizer's 600. F1 remains comparable, runtime/GFLOPs score is not strictly equivalent.")

loader = DataLoader(TensorDataset(input_ids, attention_mask),
                    batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
from torch.profiler import profile

predicted_batches = []
total_seconds = 0.0
total_gflops = 0.0
counted = 0

# Reference-style: 10 full forward passes per batch with autocast+profiler.
# CUDA synchronization is added for accurate elapsed time.
for input_batch, mask_batch in loader:
    input_batch = input_batch.to("cuda", non_blocking=True)
    mask_batch = mask_batch.to("cuda", non_blocking=True)
    torch.cuda.synchronize()
    start = time.perf_counter()

    with torch.inference_mode(), autocast_context():
        with profile(with_flops=True) as prof:
            for _ in range(REPETITIONS):
                outputs = model(input_ids=input_batch, attention_mask=mask_batch)
        torch.cuda.synchronize()

    elapsed = time.perf_counter() - start
    total_seconds += elapsed
    total_gflops += sum(float(item.flops or 0) for item in prof.key_averages()) / 1e9

    probs = torch.sigmoid(outputs.logits)
    predicted_batches.append((probs > 0.5).to(torch.int64).cpu().numpy())
    counted += input_batch.shape[0]

    if counted % 3000 < BATCH_SIZE or counted == len(targets):
        print("Evaluated:", counted, "/", len(targets))

    del input_batch, mask_batch, outputs, probs, prof
    torch.cuda.empty_cache()

predictions = np.concatenate(predicted_batches, axis=0)
assert predictions.shape == targets.shape
print("Completed all public validation rows:", len(predictions))

In [ ]:
scores = {}
for col, name in enumerate(["vulnerability", "technical_debt"]):
    p, r, f1, _ = precision_recall_fscore_support(
        targets[:, col], predictions[:, col],
        average="binary", zero_division=0
    )
    scores[name] = {
        "precision": float(p), "recall": float(r), "f1": float(f1),
        "confusion_matrix": confusion_matrix(
            targets[:, col], predictions[:, col], labels=[0, 1]
        ).tolist()
    }

mean_f1 = (scores["vulnerability"]["f1"] + scores["technical_debt"]["f1"]) / 2

# The reference's reported 'GFLOPS' is a profiler-counted operation total,
# divided by repeat count; it is not a FLOPs-per-second throughput.
avg_runtime_seconds = total_seconds / REPETITIONS
avg_gflops = total_gflops / REPETITIONS
MAX_RUNTIME = 1000
MAX_GFLOPS = 3000000
reference_formula_score = (
    0.60 * mean_f1
    + 0.20 * max((MAX_RUNTIME - avg_runtime_seconds) / MAX_RUNTIME, 0)
    + 0.20 * max((MAX_GFLOPS - avg_gflops) / MAX_GFLOPS, 0)
)

record = {
    "status": "local reference-style Colab T4 evaluation; not organizer-hidden-test",
    "model": MODEL_ID, "dataset": DATASET_ID, "public_split": SPLIT,
    "samples": int(len(targets)), "gpu": GPU_NAME,
    "settings": {
        "max_length": MAX_LENGTH, "global_padded_length": PADDED_LENGTH,
        "batch_size": BATCH_SIZE, "reference_batch_size": REFERENCE_BATCH_SIZE,
        "repetitions": REPETITIONS, "mixed_precision": "fp16 autocast",
        "thresholds": [0.5, 0.5], "cuda_sync_added": True
    },
    "tasks": scores, "mean_f1": float(mean_f1),
    "total_elapsed_seconds_for_all_repetitions": total_seconds,
    "avg_runtime_seconds_per_full_public_split": avg_runtime_seconds,
    "avg_profiler_counted_gflops_per_full_public_split": avg_gflops,
    "reference_formula_score_local_estimate": reference_formula_score,
    "limitations": [
        "Public validation, not hidden test",
        "The organizer reference uses no CUDA synchronization around its timer",
        "Batch size may fall below 600 after an OOM, affecting runtime and compute comparison",
        "Profiler-counted GFLOPs are not GFLOPs per second and omit some operations",
        "Only organizer evaluation determines competition ranking"
    ]
}
OUTPUT.write_text(json.dumps(record, indent=2), encoding="utf-8")
print(json.dumps(record, indent=2))
display(FileLink(str(OUTPUT)))

## Interpretation and next actions

1. Compare **public-validation Mean F1 at 512 tokens** against E9's independently replicated **256-token Mean F1 = 0.7830049253**. They are different evaluation protocols and should be labeled as such.
2. To investigate the released baseline discrepancy, **copy this notebook** and change `MODEL_ID` to `NLBSE/nlbse27-code-comment-classification-baseline`. Use the **same max length and batching**. The official repo reports about 0.910 baseline Mean F1, while a previous participant 256-token local run measured 0.7952; do not assume the difference is entirely caused by max length until tested.
3. Download the generated `E9_Colab_T4_reference_style_metrics.json` and archive it in GitHub.
4. The final ranking requires organizer hidden-test evaluation. Do not label this local reference-style estimate an official score.

**Reference:** [Organizer reference baseline notebook](https://colab.research.google.com/drive/1q1fGPpU7uxaK1mUVHa3b2xJOtEwhWwLG).